In [1]:
level_depth = 1

In [2]:
import os
import argparse
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio

# Headless-safe Plotly renderer
pio.renderers.default = "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"


print(f"[INFO] level_depth={level_depth}")

# --- Per-level layout only ---
BASE_LEVEL_DIR = Path(f"/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_{level_depth}")
GRID_DIR       = BASE_LEVEL_DIR / "grid_search"

DF_SAMPLED_CSV = GRID_DIR / f"df_sampled_level_{level_depth}.csv"
GRID_RESULTS   = GRID_DIR / f"grid_search_results_level_{level_depth}.csv"
MODELS_DIR     = GRID_DIR / f"bertopic_models_level_{level_depth}"

# --- Validate required inputs ---
missing = [p for p in (DF_SAMPLED_CSV, GRID_RESULTS, MODELS_DIR) if not p.exists()]
if missing:
    miss_str = "\n  - ".join(str(m) for m in missing)
    raise FileNotFoundError(f"Required paths/files not found for level {level_depth}:\n  - {miss_str}")

print(f"[INFO] DF_SAMPLED_CSV = {DF_SAMPLED_CSV}")
print(f"[INFO] GRID_RESULTS   = {GRID_RESULTS}")
print(f"[INFO] MODELS_DIR     = {MODELS_DIR}")

# --- Load dataframes ---
df_sampled = pd.read_csv(DF_SAMPLED_CSV)
df_grid = pd.read_csv(GRID_RESULTS)

# --- Compute avg_score and filter (aligned with your grid search) ---
required_cols = ("silhouette", "coherence", "diversity", "hdbscan_min_cluster_size")
missing_cols = [c for c in required_cols if c not in df_grid.columns]
if missing_cols:
    raise ValueError(f"Missing columns in grid_search_results: {missing_cols}")

df_grid["avg_score"] = (df_grid["silhouette"] + df_grid["coherence"] + df_grid["diversity"]) / 3
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- Select best rows for each metric ---
best_models = {
    "silhouette": df_grid[
        (df_grid["hdbscan_min_cluster_size"] != 10) &
        (df_grid["umap_n_neighbors"] != 25) &
        (df_grid["umap_n_neighbors"] != 3)
    ].sort_values(by="silhouette", ascending=False).iloc[0],

    # Più topic: preferisci cluster più piccoli (10 o 15), UMAP più locale (5) e embedding più compatti (min_dist=0.0)
    "coherence": df_grid[
        (df_grid["hdbscan_min_cluster_size"].isin([10, 30])) &
        (df_grid["umap_n_neighbors"]  != 25) &
        (df_grid["umap_min_dist"] == 0.0)
    ].sort_values(by="coherence", ascending=False).iloc[0],

    "diversity": df_grid[
        (df_grid["hdbscan_min_cluster_size"].isin([10, 30])) &
        (df_grid["umap_n_neighbors"] != 25) &
        (df_grid["umap_min_dist"] == 0.0)
    ].sort_values(by="diversity", ascending=False).iloc[0],

    "avg_score": df_grid[
        (df_grid["hdbscan_min_cluster_size"].isin([10, 30])) &
        (df_grid["umap_n_neighbors"]  != 25) &
        (df_grid["umap_min_dist"] == 0.0)
    ].sort_values(by="avg_score", ascending=False).iloc[0],
}




# Expose for later cells
print("[INFO] Data and grid loaded. Variables available:")
print(f"  MODELS_DIR = {MODELS_DIR}")
print(f"  DF_SAMPLED_CSV = {DF_SAMPLED_CSV}")
print(f"  GRID_RESULTS = {GRID_RESULTS}")
print("  best_models keys =", list(best_models.keys()))

print("df_sampled sampled")
print(df_sampled.head())


/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] level_depth=1
[INFO] DF_SAMPLED_CSV = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/df_sampled_level_1.csv
[INFO] GRID_RESULTS   = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/grid_search_results_level_1.csv
[INFO] MODELS_DIR     = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1
[INFO] Data and grid loaded. Variables available:
  MODELS_DIR = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1
  DF_SAMPLED_CSV = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/df_sampled_level_1.csv
  GRID_RESULTS = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/grid_search_results_level_1.csv
  best_models keys = ['silhouette', 'coherence', 'diversity', 'avg_score']
df_sampled sampled
                          

In [3]:
#SILHOUETTE

from pathlib import Path
from bertopic import BERTopic

print("[INFO] Visualizzo il best model per SILHOUETTE")

key = "silhouette"
row = best_models[key]

suffix = (
    f"{row['model']}_"
    f"umap{row['umap_n_components']}_"
    f"umap{row['umap_n_neighbors']}_"
    f"umap{row['umap_min_dist']}_"
    f"hdbscan{row['hdbscan_min_cluster_size']}"
)
model_path = Path(MODELS_DIR) / suffix
print(model_path)

if not model_path.exists():
    print(f"[WARN] Path not found: {model_path}")
else:
    print(f"[INFO] Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(str(model_path))
    num_topics = len(set(topic_model.topics_))
    print(f"[INFO] {key}: {num_topics} topics found, at most 180 topics are shown")

    fig = topic_model.visualize_barchart(
        top_n_topics=min(180, num_topics),
        n_words=20,
        width=350,
        height=450
    )
    try:
        fig.show()
    except Exception:
        print("[WARN] Impossibile mostrare il grafico in questa sessione.")


[INFO] Visualizzo il best model per SILHOUETTE
/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/all-distilroberta-v1_umap5_umap5_umap0.0_hdbscan30
[INFO] Loading model for best silhouette from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/all-distilroberta-v1_umap5_umap5_umap0.0_hdbscan30
[INFO] silhouette: 958 topics found, at most 180 topics are shown


In [4]:
#COHERENCE from pathlib import Path

from bertopic import BERTopic

print("[INFO] Visualizzo il best model per COHERENCE")

key = "coherence"
row = best_models[key]

suffix = (
    f"{row['model']}_"
    f"umap{row['umap_n_components']}_"
    f"umap{row['umap_n_neighbors']}_"
    f"umap{row['umap_min_dist']}_"
    f"hdbscan{row['hdbscan_min_cluster_size']}"
)
model_path = Path(MODELS_DIR) / suffix
print(model_path)

if not model_path.exists():
    print(f"[WARN] Path not found: {model_path}")
else:
    print(f"[INFO] Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(str(model_path))
    num_topics = len(set(topic_model.topics_))
    print(f"[INFO] {key}: {num_topics} topics found, at most 180 topics are shown")

    fig = topic_model.visualize_barchart(
        top_n_topics=min(180, num_topics),
        n_words=20,
        width=350,
        height=450
    )
    try:
        fig.show()
    except Exception:
        print("[WARN] Impossibile mostrare il grafico in questa sessione.")


[INFO] Visualizzo il best model per COHERENCE
/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/paraphrase-MiniLM-L6-v2_umap10_umap5_umap0.0_hdbscan30
[INFO] Loading model for best coherence from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/paraphrase-MiniLM-L6-v2_umap10_umap5_umap0.0_hdbscan30
[INFO] coherence: 873 topics found, at most 180 topics are shown


In [5]:
#DIVERSITY

from pathlib import Path
from bertopic import BERTopic

print("[INFO] Visualizzo il best model per DIVERSITY")

key = "diversity"
row = best_models[key]

suffix = (
    f"{row['model']}_"
    f"umap{row['umap_n_components']}_"
    f"umap{row['umap_n_neighbors']}_"
    f"umap{row['umap_min_dist']}_"
    f"hdbscan{row['hdbscan_min_cluster_size']}"
)
model_path = Path(MODELS_DIR) / suffix
print(model_path)

if not model_path.exists():
    print(f"[WARN] Path not found: {model_path}")
else:
    print(f"[INFO] Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(str(model_path))
    num_topics = len(set(topic_model.topics_))
    print(f"[INFO] {key}: {num_topics} topics found, at most 180 topics are shown")

    fig = topic_model.visualize_barchart(
        top_n_topics=min(180, num_topics),
        n_words=20,
        width=350,
        height=450
    )
    try:
        fig.show()
    except Exception:
        print("[WARN] Impossibile mostrare il grafico in questa sessione.")


[INFO] Visualizzo il best model per DIVERSITY
/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/paraphrase-MiniLM-L6-v2_umap3_umap5_umap0.0_hdbscan30
[INFO] Loading model for best diversity from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/paraphrase-MiniLM-L6-v2_umap3_umap5_umap0.0_hdbscan30
[INFO] diversity: 867 topics found, at most 180 topics are shown


In [6]:
#AVG_SCORE

from pathlib import Path
from bertopic import BERTopic

print("[INFO] Visualizzo il best model per AVG_SCORE")

key = "avg_score"
row = best_models[key]

suffix = (
    f"{row['model']}_"
    f"umap{row['umap_n_components']}_"
    f"umap{row['umap_n_neighbors']}_"
    f"umap{row['umap_min_dist']}_"
    f"hdbscan{row['hdbscan_min_cluster_size']}"
)
model_path = Path(MODELS_DIR) / suffix
print(model_path)

if not model_path.exists():
    print(f"[WARN] Path not found: {model_path}")
else:
    print(f"[INFO] Loading model for best {key} from: {model_path}")
    topic_model = BERTopic.load(str(model_path))
    num_topics = len(set(topic_model.topics_))
    print(f"[INFO] {key}: {num_topics} topics found, at most 180 topics are shown")

    fig = topic_model.visualize_barchart(
        top_n_topics=min(180, num_topics),
        n_words=20,
        width=350,
        height=450
    )
    try:
        fig.show()
    except Exception:
        print("[WARN] Impossibile mostrare il grafico in questa sessione.")


[INFO] Visualizzo il best model per AVG_SCORE
/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/all-distilroberta-v1_umap5_umap5_umap0.0_hdbscan30
[INFO] Loading model for best avg_score from: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_1/grid_search/bertopic_models_level_1/all-distilroberta-v1_umap5_umap5_umap0.0_hdbscan30
[INFO] avg_score: 958 topics found, at most 180 topics are shown


In [7]:
#SILHOUETTE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['silhouette'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


NameError: name 'topic_models' is not defined

In [ ]:
topic_models['silhouette'].get_topic_info()

In [ ]:
topic_models['silhouette'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

In [ ]:
df_sampled['topic']=topic_models['silhouette'].topics_

In [ ]:
#COHERERENCE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['coherence'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [ ]:
topic_models['coherence'].get_topic_info()

In [ ]:
topic_models['coherence'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

In [ ]:
df_sampled['topic']=topic_models['coherence'].topics_

In [ ]:
#DIVERSITY

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['diversity'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [ ]:
topic_models['diversity'].get_topic_info()

In [ ]:
topic_models['diversity'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

In [ ]:
df_sampled['topic']=topic_models['diversity'].topics_

In [ ]:
#AVG_SCORE

texts = df_sampled['text_preprocessed']
timestamps = pd.to_datetime(df_sampled['timestamp'], unit='s').dt.date
timestamps = pd.to_datetime(timestamps)

topics_over_time = topic_models['avg_score'].topics_over_time(
    docs=list(texts),
    timestamps=list(timestamps),
    nr_bins = 52,
    global_tuning = False,
    evolution_tuning = True)

topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=-1)


In [ ]:
topic_models['avg_score'].get_topic_info()

In [ ]:
topic_models['avg_score'].get_topic_info().set_index('Topic').loc[1,'Representative_Docs']

In [ ]:
df_sampled['topic']=topic_models['avg_score'].topics_